# Writing data to and reading data from a Database using Python

## Libraries and settings

In [22]:
# Libraries
import os
import sqlite3
import fnmatch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Function to close a sqlite db-connection
def check_conn(conn):
     try:
        conn.cursor()
        return True
     except Exception as ex:
        return False

# Get current working directory
print(os.getcwd())

/workspaces/applied_research_methods/Week_02


## Create sqlite data base

In [23]:
# Create data base
conn = sqlite3.connect('./Data/supermarkets_database.db') 
cursor = conn.cursor()

# Show dbs in the directory
flist = fnmatch.filter(os.listdir('.'), '*.db')
for i in flist:
    print(i)

## Create SQL-table in the database

In [24]:
cursor.execute('''CREATE TABLE IF NOT EXISTS supermarkets_table (supermarket VARCHAR(50),
                                                                 city VARCHAR(100),
                                                                 housenumber INT(5),
                                                                 postcode INT(4), 
                                                                 street VARCHAR(100))''')
# Confirm changes to the table
conn.commit()

## Read data from file to data frame

In [25]:
# Import json file
df = pd.read_json('./Data/supermarkets.json', encoding='utf-8')

# Change to data frame
df2 = pd.DataFrame.from_records(df.tags)
df2 = df2[['brand', 'addr:city', 'addr:street', 'addr:housenumber', 'addr:postcode']]

# Rename selected columns
df2 = df2.rename(columns={'brand': 'supermarket',
                         'addr:city': 'city',
                         'addr:housenumber': 'housenumber',
                         'addr:postcode': 'postcode',
                         'addr:street':'street'})
df2.head(5)


,supermarket,city,street,housenumber,postcode
0,Spar,NaN,NaN,NaN,NaN
1,Migros,Uznach,Zürcherstrasse,25,8730
2,Coop,Uznach,NaN,NaN,8730
3,Coop,Zürich,Bahnhofbrücke,1,8001
4,Migros,Zürich,Wengistrasse,7,8004


## Write data to the SQL-table in data base

In [26]:
df2.to_sql(name = 'supermarkets_table',
          con = conn,
          index = False,
          if_exists = 'replace')

3392

## Query the SQL-table

In [ ]:
# Query the SQL-table
cursor.execute('''SELECT *
               FROM supermarkets_table
               WHERE upper(city) like upper('Winterthur') ''')
# with upper() (or lower()) making sure it is case insensitive

df = pd.DataFrame(cursor.fetchall(), 
                  columns=['supermarket','city','housenumber','postcode','street'])    
df

,supermarket,city,housenumber,postcode,street
0,Migros,Winterthur,Zürcherstrasse,102,8406
1,Migros,Winterthur,Schaffhauserstrasse,152,8400
2,Migros,Winterthur,Wülflingerstrasse,71,8400
3,None,Winterthur,Frauenfelderstrasse,69,8404
4,None,Winterthur,Bankstrasse,8/12,8400
5,None,Winterthur,Steinberggasse,18,8400
6,Migros,Winterthur,Hinterdorfstrasse,40,8405
7,Denner,Winterthur,Hinterdorfstrasse,40,8405
8,Migros,Winterthur,Strickerstrasse,3,8400
9,Migros,Winterthur,Stadthausstrasse,31,8400


## Close db connection (if open)

In [20]:
# Close db connection (if open)
try:
    if check_conn(conn):
        conn.close()
    else:
        pass
except:
    pass

# Status (True = open, False = closed)
print(check_conn(conn))

False


### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [21]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.5.0-1025-azure
Datetime: 2024-11-08 10:24:20
Python Version: 3.11.10
-----------------------------------
